# 🧠 NeuralAI — SDXL Image LoRA Training (Google Colab)

Train a brand **"vibe stack"** SDXL LoRA (dark mode, neon, cinematic) using the official **diffusers** SDXL LoRA trainer on a free Colab T4 GPU.

**Why this notebook (not Unsloth Studio):** Unsloth Studio's no-code Train UI only supports Text / Vision / Audio / Embeddings / BERT models — it cannot train diffusion (SDXL) models. So we use the official `diffusers` `train_text_to_image_lora_sdxl.py` script, which drops straight into `services/diffusion_engine.py` (`NeuralAIDiffusion`) via `pipe.load_lora_weights(...)`.

**Workflow:**
1. Install diffusers stack
2. Upload 20–50 brand images (the style source)
3. Build the dataset (image + caption)
4. Train the LoRA (scripted, ~10 epochs)
5. Test generation
6. Export & wire into the live sidecar via `NEURALAI_LORA_PATH`

## 1. Install diffusers + SDXL training stack

Run on a **Colab GPU runtime** (T4 or better).

In [ ]:
# Install the diffusers SDXL LoRA training stack
!pip install --quiet --upgrade diffusers transformers accelerate peft pillow torchvision datasets

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Switch to a GPU runtime: Runtime > Change runtime type > GPU")

## 2. Upload brand images

The LoRA learns the *style* from **real images**. Upload **20–50** brand images (`.png`/`.jpg`). Optionally include a `image.txt` next to any image to override its caption. You can also mount Google Drive and copy images into `images/`.

In [ ]:
from pathlib import Path
from google.colab import files

img_dir = Path("images"); img_dir.mkdir(exist_ok=True)
print("Select 20-50 brand images (png/jpg). A file picker will appear.")
uploaded = files.upload()  # opens the picker
for name in uploaded.keys():
    src = Path(name)
    if src.exists():
        src.rename(img_dir / name)
print("Images in images/:", [p.name for p in sorted(img_dir.iterdir())])

In [ ]:
# OPTIONAL: mount Google Drive and copy images from a Drive folder into images/
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# src = Path("/content/drive/MyDrive/NeuralAI/brand_images")
# for f in src.glob("*"):
#     if f.suffix.lower() in (".png",".jpg",".jpeg",".webp"):
#         shutil.copy(f, img_dir / f.name)
# print("Images now in images/:", len(list(img_dir.iterdir())))

## 3. Build the dataset

Copies images into `neuralai_lora_data/` and writes `metadata.jsonl` (`file_name` + `text`). Each image is captioned with the brand style prompt (or its sidecar `image.txt`).

In [ ]:
import json, shutil
from pathlib import Path

src = Path("images")
out = Path("neuralai_lora_data"); out.mkdir(exist_ok=True)
BRAND = ("neuralai vibe stack style, cinematic dark mode, neon accent lighting, "
         "high contrast, hyper-detailed, 8k, volumetric fog, professional concept art")

rows = []
for img in sorted(src.iterdir()):
    if img.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp"):
        shutil.copy(img, out / img.name)
        caption = BRAND
        sidecar = out / (img.stem + ".txt")
        if sidecar.exists():
            caption = sidecar.read_text().strip() or BRAND
        rows.append({"file_name": img.name, "text": caption})

if not rows:
    raise SystemExit("No images found in images/ — run the upload cell first.")

with open(out / "metadata.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")
print(f"Dataset ready: {len(rows)} images -> {out / 'metadata.jsonl'}")

## 4. Train the SDXL LoRA

Downloads the official `train_text_to_image_lora_sdxl.py` and runs it. Output goes to `neuralai_sdxl_lora/` as `pytorch_lora_weights.safetensors` (loads directly via `pipe.load_lora_weights`).

In [ ]:
# Fetch the official SDXL LoRA training script
!curl -L -o train_text_to_image_lora_sdxl.py https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py

# Train (T4-friendly: fp16, batch 2, res 1024, rank 16, 10 epochs)
!python train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --train_data_dir="./neuralai_lora_data" \
  --output_dir="./neuralai_sdxl_lora" \
  --resolution=1024 --center_crop --random_flip \
  --train_batch_size=2 \
  --num_train_epochs=10 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" --lr_warmup_steps=0 \
  --mixed_precision="fp16" \
  --seed=42 \
  --rank=16 \
  --validation_prompt="a sentient AI core glowing in a dark server room, neuralai vibe stack style" \
  --validation_epochs=5

## 5. Test generation

Loads the base SDXL model + your trained LoRA and generates a test image.

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline

BASE = "stabilityai/stable-diffusion-xl-base-1.0"
pipe = StableDiffusionXLPipeline.from_pretrained(BASE, torch_dtype=torch.float16)
pipe.to("cuda")
pipe.load_lora_weights("neuralai_sdxl_lora")

prompt = "a sentient AI core glowing in a dark server room, neuralai vibe stack style"
img = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
img.save("neuralai_test.png")
img

## 6. Export & wire into the live sidecar

The trained adapter is in `neuralai_sdxl_lora/` (`pytorch_lora_weights.safetensors`). To use it on the NeuralAI host (ZO Computer / local):

```bash
# On the NeuralAI host
export NEURALAI_DIFFUSION=1
export NEURALAI_LORA_PATH=/path/to/neuralai_sdxl_lora
```

`services/diffusion_engine.py` already calls `pipe.load_lora_weights(NEURALAI_LORA_PATH)`, so the brand LoRA applies automatically on every local generation. Download the `neuralai_sdxl_lora/` folder from Colab (Files sidebar > right-click > Download) and place it on the host.